# Sesion 7 MongoDb Avanzado
Curso de Especialización en Inteligencia Artificial y Big Data
Profesor: Juan Carlos Pérez González
Departamento de Informática – IES de Teis

**Replicaset (replicación)**

**Concepto**: conjunto de nodos que mantienen copias idénticas de los mismos datos.

**Objetivo**: alta disponibilidad y tolerancia a fallos.

1. Un nodo es **primario**, permite **escritura**.

2. Los demás son **secundarios** y **replican datos del primario**.

3. Si el primario falla, **uno de los secundarios se convierte en primario** automáticamente.

Esto permite protege tus datos de pérdida y permite lecturas distribuidas.

IMPORTANTE: vamos a trabajar sin autenticación. No es lo que hay que hacer pero en Windows, los docker para trabajar con Replica-Set dan bastante problemas.
En los apuntes esta la versión con autenticación con MV de Ubuntu que no da problemas. 

## Trabajando con Replica-sets 

**Paso 1**. Creamos dentro de replicaset la carpeta data y dentrod de ella mongo1, mongo2 y mongo3, desde **powershell como administrador**

Si miras el fichero docker mongo-replicaset lo entiendes. 

*New-Item -ItemType Directory -Force data\mongo1*

*New-Item -ItemType Directory -Force data\mongo2*

*New-Item -ItemType Directory -Force data\mongo3*


**Paso 2**. Ejecutamos el docker-compose replicaset que tenemos en 4.2.Almacen  **mongo-replica.set** fuera del directorio mongo-keyfaile pero dentro del directorio replicaset.

*docker network create mongo-replicaset-net*

*docker-compose -f mongo-replicaset.yml up -d*


**Paso 3.** Iniciamos la replica. Para ello nos conectamos al mongodb1

Así sería con autenticación:
*docker exec -it mongodb1 mongosh -u admin -p abc123 --authenticationDatabase admin*

Pero como no tenemos autenticación **usamos este comando**:

*docker exec -it mongodb1 mongo*


y dentro de mongosh iniciamos la replicaset


*rs.initiate({
  _id: "rs0",           
  members: [
    { _id: 0, host: "mongodb1:27017" },
    { _id: 1, host: "mongodb2:27017" },
    { _id: 2, host: "mongodb3:27017" }
  ]
})*


*rs0* es el nombre del replicaset que le dimos en docker-compose

Comprobamos que todo va bien:

*rs.status()*

Vamos a conectarnos desde jupyterlab

In [1]:
from pymongo import MongoClient

cliente = MongoClient('mongodb1', 27017, replicaset='rs0')
db = cliente.test
print(db.list_collection_names())




ModuleNotFoundError: No module named 'pymongo'

In [ ]:
#Comprobamos que ha descubierto automáticamente el resto de nodos
print(cliente.nodes)

In [ ]:
import pandas as pd

# Cargar CSV
df = pd.read_csv("RegistroComprasOnline.csv")

# Ver las primeras filas
print(df.head())  # <- aquí agregamos print

# Conexión a la base y colección
db = cliente['bbdd']
coleccion = db['compras']

# Convertir a diccionarios e insertar
registros = df.to_dict(orient='records')
resultado = coleccion.insert_many(registros)

print(f"Se insertaron {len(resultado.inserted_ids)} registros.")


In [ ]:
#Hacemos una consulta
# Leer los primeros 10 documentos
primeros_10 = coleccion.find().limit(10)

# Imprimirlos
for doc in primeros_10:
    print(doc)

In [ ]:
#Una vez parado el primario, comprobamos los nodos reconocidos
#Comprobamos que ha descubierto automáticamente el resto de nodos
print(cliente.nodes)

In [ ]:
#Repetimos la consulta para verificar que podemos seguir trabajando
print(coleccion.find_one())

In [ ]:
#Y hacemos una inserción para comprobar que hay un nuevo primario
# (si no hubiese primario no podrías modificar)
resultado = coleccion.insert_one({'IDCliente': 34, 'test': 1})
print(resultado.inserted_id)

Nos conectamos a otro mongo y ves que se replicaron los datos

In [ ]:
# Conectarse a mongodb2
cliente_mongo2 = MongoClient(
    'mongodb2',       # nombre del contenedor dentro de la red
    27017,            # puerto interno de Mongo
    replicaset='rs0', # replicaset que configuraste
    readPreference='secondaryPreferred'
)

# Seleccionamos la base y la colección
db_m2 = cliente_mongo2['bbdd']
coleccion_m2 = db_m2['compras']

# Leer 10 primeros documentos
for doc in coleccion_m2.find().limit(10):
    print(doc)


Lo mismo con el otro nodo 

In [ ]:
from pymongo import MongoClient

# Conectarse a mongodb2
cliente_mongo3 = MongoClient(
    'mongodb3',       # nombre del contenedor dentro de la red
    27017,            # puerto interno de Mongo
    replicaset='rs0', # replicaset que configuraste
    readPreference='secondaryPreferred'
)

# Seleccionamos la base y la colección
db_m3 = cliente_mongo3['bbdd']
coleccion_m3 = db_m3['compras']

# Leer 10 primeros documentos
for doc in coleccion_m3.find().limit(10):
    print(doc)


Ahora vamos a parar mongodb1 y comprobar que otro coge el papel de primario. Podemos hacerlo con Docker

A continuación nos conectamos, por ejemplo, mongodb2

*docker exec -it mongodb2 mongo*

Y comprobamos que cual es el primario y podemos hacer una inserción"

*rs.status()*


In [ ]:
from pymongo import MongoClient
from pymongo.errors import OperationFailure

# --- Conexión al primario (mongodb2) ---
cliente_primario = MongoClient('mongodb2', 27017,
                               replicaset='rs0',
                               readPreference='primary')

db_primario = cliente_primario['bbdd']
coleccion_primario = db_primario['compras']

# Intentamos insertar un documento
doc_primario = {"producto": "TestPrimario", "cantidad": 5}

resultado_primario = coleccion_primario.insert_one(doc_primario)
print(f"Inserción en primario OK, id: {resultado_primario.inserted_id}")


# --- Conexión al secundario (mongodb3) ---
cliente_secundario = MongoClient('mongodb3', 27017,
                                 replicaset='rs0',
                                 readPreference='secondary')

db_secundario = cliente_secundario['bbdd']
coleccion_secundario = db_secundario['compras']

# Intentamos insertar un documento en secundario
doc_secundario = {"producto": "TestSecundario", "cantidad": 10}

try:
    resultado_secundario = coleccion_secundario.insert_one(doc_secundario)
    print(f"Inserción en secundario OK, id: {resultado_secundario.inserted_id}")
except OperationFailure as e:
    print(f"No se pudo insertar en secundario: {e}")


**Muy importante**, lo que está pasando no es que haya insertado en el secundario sino que dirigió la inserción al primario, pero si hago una conexión direrecat sin replicaset así:

In [ ]:
from pymongo import MongoClient
from pymongo.errors import OperationFailure

# Conexión directa al secundario dentro del replicaset e intentar escribir
cliente_sec = MongoClient(
    'mongodb3', 
    27017, 
    replicaset='rs0',     # indicamos replicaset
    directConnection=True  # fuerza conexión directa a este nodo
)
db_sec = cliente_sec['bbdd']
coleccion_sec = db_sec['compras']

try:
    coleccion_sec.insert_one({"test": 123})
except OperationFailure as e:
    print("Fallo al intentar escribir directamente en secundario:", e)


# Sharding

## Conexión a mongos

In [ ]:
from pymongo import MongoClient

#Utiliza la IP de tu anfitrión
cliente = MongoClient('mongos', 27017)

#Creamos la instancia para interactuar con la colección de clientes y pedidos
bbdd = cliente.tienda
coleccion_clientes = bbdd.clientes
coleccion_pedidos = bbdd.pedidos

## Inserción de datos de clientes y de pedidos

In [ ]:
#Inserción de documentos de clientes
documentos_clientes = [
    {'IDCliente': 1, 'Nombre': "Juan", 'Apellidos': "Fernández"},
    {'IDCliente': 2, 'Nombre': "María", 'Apellidos': "Fernández"},
    {'IDCliente': 3, 'Nombre': "Carolina", 'Apellidos': "Pérez"}
]

resultado = coleccion_clientes.insert_many(documentos_clientes)
print("Se han insertado",len(resultado.inserted_ids),"clientes")

In [ ]:
#Inserción de documentos de pedidos
documentos_pedidos = [
    {'IDPedido': 1, 'IDCliente': 1, 'Importe': 3.24, 'Ciudad': "Vigo"},
    {'IDPedido': 2, 'IDCliente': 1, 'Importe': 8.01, 'Ciudad': "Pontevedra"},
    {'IDPedido': 3, 'IDCliente': 3, 'Importe': 28.12, 'Ciudad': "A Coruña"},
    {'IDPedido': 4, 'IDCliente': 1, 'Importe': 56.78, 'Ciudad': "Vigo"},
    {'IDPedido': 5, 'IDCliente': 2, 'Importe': 0.12, 'Ciudad': "Madrid"},
    {'IDPedido': 6, 'IDCliente': 3, 'Importe': 99.45, 'Ciudad': "Barcelona"},
    {'IDPedido': 7, 'IDCliente': 3, 'Importe': 2.1, 'Ciudad': "Valencia"},
    {'IDPedido': 8, 'IDCliente': 1, 'Importe': 9, 'Ciudad': "Ourense"},
    {'IDPedido': 9, 'IDCliente': 1, 'Importe': 32.56, 'Ciudad': "Lugo"},
    {'IDPedido': 10, 'IDCliente': 3, 'Importe': 5.45, 'Ciudad': "Santiago"},
]

resultado = coleccion_pedidos.insert_many(documentos_pedidos)
print("Se han insertado",len(resultado.inserted_ids),"pedidos")

## Comprobación de datos insertados

In [ ]:
#Número de documentos en colección clientes
num_clientes = coleccion_clientes.count_documents({})
print("Hay",num_clientes,"clientes")

#Número de documentos en colección pedidos
num_pedidos = coleccion_pedidos.count_documents({})
print("Hay",num_pedidos,"pedidos")

## Conexión a mongoshard1 y cuenta de documentos

In [ ]:
from pymongo import MongoClient

#Como JupyterLab y los nodos de Mongo están conectados a través de la red interna
# establecemos la conexión a travñes de dicha red. Todos escuchan en esa red en el puerto 27017.
# Si te quisieses conectar a través del anfitrión, debes hacerlo al puerto mapeado, en esta caso
# al 27019
cliente = MongoClient('mongoshard1', 27017)

#Creamos la instancia para interactuar con la colección de clientes y pedidos
bbdd = cliente.tienda
coleccion_clientes = bbdd.clientes
coleccion_pedidos = bbdd.pedidos

#Número de documentos en colección clientes
num_clientes = coleccion_clientes.count_documents({})
print("Hay",num_clientes,"clientes")

#Número de documentos en colección pedidos
num_pedidos = coleccion_pedidos.count_documents({})
print("Hay",num_pedidos,"pedidos")

## Conexión a mongoshard2 y cuenta de documentos

In [ ]:
from pymongo import MongoClient

#Como JupyterLab y los nodos de Mongo están conectados a través de la red interna
# establecemos la conexión a travñes de dicha red. Todos escuchan en esa red en el puerto 27017.
# Si te quisieses conectar a través del anfitrión, debes hacerlo al puerto mapeado, en esta caso
# al 27020
cliente = MongoClient('mongoshard2', 27017)

#Creamos la instancia para interactuar con la colección de clientes y pedidos
bbdd = cliente.tienda
coleccion_clientes = bbdd.clientes
coleccion_pedidos = bbdd.pedidos

#Número de documentos en colección clientes
num_clientes = coleccion_clientes.count_documents({})
print("Hay",num_clientes,"clientes")

#Número de documentos en colección pedidos
num_pedidos = coleccion_pedidos.count_documents({})
print("Hay",num_pedidos,"pedidos")